# Multimodal AI Pipelines

Companion notebook for **IM Session 38 - Multimodal AI Pipelines**.

We build a **voice-note → text → fields → decision → spoken reply** pipeline with a quality gate between every stage, then apply the same gate idea to images.

Everything here is real and runs locally: `whisper-tiny.en` for speech-to-text (downloads once, ~150 MB), macOS `say` for text-to-speech, Pillow for images. An optional local LLM step (Ollama) is guarded and skipped if Ollama is not running.

**Dataset** (in the `Datasets` folder, or next to this notebook): `voice_note_1..3.wav`, `voice_note_silent.wav`, `voice_notes_reference.csv`, `receipt.png`, `receipt_too_dark.png`, `blank_page.png`. The voice notes were synthesised from the reference sentences, so errors can be measured exactly.

In [1]:
import os, re, difflib, shutil, subprocess, tempfile, time, wave
import numpy as np
import pandas as pd
from PIL import Image

DATA = "." if os.path.exists("voice_note_1.wav") else "../Datasets"
path = lambda name: os.path.join(DATA, name)
ref = pd.read_csv(path("voice_notes_reference.csv"))
ref

,file,reference_transcript
0,voice_note_1.wav,Delivery in Gurugram was late by forty minutes...
1,voice_note_2.wav,Please change the delivery address for order 4...
2,voice_note_3.wav,The store in Chennai had a damaged box and the...


## 1. Stage one — speech-to-text

Load a small open-source speech model and transcribe every clip. Results are cached so each clip is transcribed once.

In [2]:
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en")
TRANSCRIPTS = {}

def transcribe(filename):
    if filename not in TRANSCRIPTS:
        TRANSCRIPTS[filename] = asr(path(filename))["text"].strip()
    return TRANSCRIPTS[filename]

for f in ["voice_note_1.wav", "voice_note_2.wav", "voice_note_3.wav"]:
    print(f, "->", transcribe(f))

/Users/suman/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/suman/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device set to use mps:0


`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


voice_note_1.wav -> delivery and guregram was late by 40 minutes. The customer asked for a refund because the order arrived cold.


voice_note_2.wav -> Please change the delivery address for order 4572 to Noida Sector 62 before its ships.


voice_note_3.wav -> The store in Chennai had a damaged box and the item inside was crushed. I want a replacement, not a refund.


## 2. Measuring the damage — word error rate (WER)

WER = word edits (substitutions + insertions + deletions) ÷ words in the reference. We normalise case, punctuation and spelled-out numbers first so formatting differences don't count as errors.

In [3]:
NUMBER_WORDS = {"forty": "40", "sixty two": "62"}

def normalise(text):
    t = re.sub(r"[^a-z0-9 ]", "", text.lower())
    for words, digits in NUMBER_WORDS.items():
        t = t.replace(words, digits)
    return t.split()

def wer(reference, hypothesis):
    r, h = normalise(reference), normalise(hypothesis)
    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
    for i in range(len(r) + 1): d[i][0] = i
    for j in range(len(h) + 1): d[0][j] = j
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            cost = 0 if r[i - 1] == h[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + cost)
    return d[len(r)][len(h)] / max(len(r), 1)

assert wer("a b c d", "a b c d") == 0 and wer("a b c d", "a x c d") == 0.25   # sanity check of the metric

rows = []
for r in ref.itertuples():
    hyp = transcribe(r.file)
    rows.append({"file": r.file, "WER": round(wer(r.reference_transcript, hyp), 3), "transcript": hyp})
wer_table = pd.DataFrame(rows)
wer_table

,file,WER,transcript
0,voice_note_1.wav,0.105,delivery and guregram was late by 40 minutes. ...
1,voice_note_2.wav,0.067,Please change the delivery address for order 4...
2,voice_note_3.wav,0.000,The store in Chennai had a damaged box and the...


Even clean synthetic speech is not transcribed perfectly. Compare note 1's transcript with its reference: **"Gurugram" became "guregram"**. Nothing flagged it.

## 3. Quality gate 1 — is the transcript usable?

Now the silent clip. A good speech model should return nothing; see what this one returns.

In [4]:
silent = transcribe("voice_note_silent.wav")
print("Length:", len(silent), "characters")
print(repr(silent[:60]), "...")

Length: 446 characters
'............................................................' ...


In [5]:
def transcript_usable(text, min_words=3):
    letters = sum(c.isalpha() for c in text)
    if letters < 0.5 * max(len(text.strip()), 1):
        return False, "mostly non-letter characters (degenerate output)"
    if len(text.split()) < min_words:
        return False, "too few words"
    return True, "ok"

for f in ["voice_note_1.wav", "voice_note_silent.wav"]:
    print(f, "->", transcript_usable(transcribe(f)))

assert transcript_usable(transcribe("voice_note_silent.wav"))[0] is False
assert all(transcript_usable(transcribe(f))[0] for f in ["voice_note_1.wav", "voice_note_2.wav", "voice_note_3.wav"])

voice_note_1.wav -> (True, 'ok')
voice_note_silent.wav -> (False, 'mostly non-letter characters (degenerate output)')


## 4. Why errors compound — and a safe repair

Extract fields from the *raw* transcript of note 1. The city is lost, silently.

In [6]:
CITIES = ["gurugram", "mumbai", "delhi", "noida", "chennai"]

def extract_city(text):
    t = text.lower()
    return next((c for c in CITIES if c in t), None)

raw1 = transcribe("voice_note_1.wav")
print("City from RAW transcript:", extract_city(raw1))

City from RAW transcript: None


Repair near-misses against the **known list of cities** using fuzzy matching. Two safeguards: a sensible cutoff, and an audit trail of every change.

In [7]:
def repair_cities(text, cutoff=0.75):
    corrections, out = [], []
    for word in text.split():
        bare = re.sub(r"[^a-z]", "", word.lower())
        match = difflib.get_close_matches(bare, CITIES, n=1, cutoff=cutoff) if bare not in CITIES else []
        if match:
            corrections.append((word, match[0].title()))
            out.append(match[0].title())
        else:
            out.append(word)
    return " ".join(out), corrections

fixed1, corrections = repair_cities(raw1)
print("Corrections:", corrections)
print("City from REPAIRED transcript:", extract_city(fixed1))

assert extract_city(raw1) is None and extract_city(fixed1) == "gurugram"

Corrections: [('guregram', 'Gurugram')]
City from REPAIRED transcript: gurugram


**Why the cutoff matters.** Too low, and ordinary words are rewritten into cities. Check that the chosen cutoff leaves every word of the other transcripts untouched.

In [8]:
for f in ["voice_note_2.wav", "voice_note_3.wav"]:
    _, c = repair_cities(transcribe(f))
    print(f, "corrections:", c)
    assert c == [], "a harmless word was wrongly 'corrected'"

_, loose = repair_cities("I will deliver the order to Delhi", cutoff=0.4)
print("With a far too low cutoff (0.4) on an innocent sentence:", loose)
assert loose != []   # the over-eager setting rewrites ordinary words — which is exactly the danger

voice_note_2.wav corrections: []
voice_note_3.wav corrections: []
With a far too low cutoff (0.4) on an innocent sentence: [('deliver', 'Delhi'), ('the', 'Chennai'), ('order', 'Noida')]


## 5. Stage two — extract fields and decide

First a naive intent rule, then the version that handles *negation*. Note 3 says "I want a replacement, **not a refund**".

In [9]:
def naive_intent(text):
    t = text.lower()
    return ("refund" if "refund" in t else "replacement" if "replacement" in t
            else "address_change" if "address" in t else "unknown")

def intent(text):
    t = re.sub(r"\bnot an? refund\b", "", text.lower())    # drop negated mentions first
    return ("refund" if "refund" in t else "replacement" if "replacement" in t
            else "address_change" if "address" in t else "unknown")

t3 = transcribe("voice_note_3.wav")
print("Naive rule :", naive_intent(t3))
print("With negation handling:", intent(t3))
assert naive_intent(t3) == "refund" and intent(t3) == "replacement"

Naive rule : refund
With negation handling: replacement


The naive rule would have queued a **refund** for a customer who explicitly refused one. A pipeline can pass every technical check and still be semantically wrong — which is why money-touching actions get a human gate.

In [10]:
def extract_fields(text):
    order = re.search(r"order\s*#?\s*(\d{3,6})", text.lower())
    return {"city": extract_city(text), "order_id": order.group(1) if order else None, "intent": intent(text)}

for f in ["voice_note_1.wav", "voice_note_2.wav", "voice_note_3.wav"]:
    fixed, _ = repair_cities(transcribe(f))
    print(f, extract_fields(fixed))

voice_note_1.wav {'city': 'gurugram', 'order_id': None, 'intent': 'refund'}
voice_note_2.wav {'city': 'noida', 'order_id': '4572', 'intent': 'address_change'}
voice_note_3.wav {'city': 'chennai', 'order_id': None, 'intent': 'replacement'}


## 6. Stage three — text-to-speech, and verifying it

macOS `say` makes the audio. A **round-trip check** transcribes the generated audio and compares it to the text we intended.

In [11]:
def speak(text, out_wav):
    if not (shutil.which("say") and shutil.which("afconvert")):
        return None
    aiff = tempfile.mktemp(suffix=".aiff")
    subprocess.run(["say", "-v", "Samantha", "-o", aiff, text], check=True)
    subprocess.run(["afconvert", "-f", "WAVE", "-d", "LEI16@16000", "-c", "1", aiff, out_wav], check=True)
    os.remove(aiff)
    return out_wav

def wav_seconds(p):
    with wave.open(p) as w:
        return w.getnframes() / w.getframerate()

intended = "Thanks. We have logged your replacement request for Chennai."
out = speak(intended, tempfile.mktemp(suffix=".wav"))
if out:
    heard = asr(out)["text"].strip()
    print(f"Audio length: {wav_seconds(out):.1f}s")
    print("Intended :", intended)
    print("Heard    :", heard)
    print("Round-trip WER:", round(wer(intended, heard), 3))
    assert wav_seconds(out) > 1.0 and wer(intended, heard) < 0.3
else:
    print("`say` not available on this machine — skipping the audio generation step.")

Audio length: 3.6s
Intended : Thanks. We have logged your replacement request for Chennai.
Heard    : Thanks. We have logged your replacement request for Chennai.
Round-trip WER: 0.0


## 7. Images — a quality gate before any vision model

A model shown a black or blank image will still answer something. Check brightness and contrast first.

In [12]:
def image_stats(filename):
    gray = np.asarray(Image.open(path(filename)).convert("L"), dtype=float)
    return gray.mean(), gray.std()

def image_usable(filename, min_brightness=60, min_contrast=20):
    brightness, contrast = image_stats(filename)
    if brightness < min_brightness:
        return False, f"too dark (brightness {brightness:.0f})"
    if contrast < min_contrast:
        return False, f"blank or washed out (contrast {contrast:.0f})"
    return True, f"ok (brightness {brightness:.0f}, contrast {contrast:.0f})"

for f in ["receipt.png", "receipt_too_dark.png", "blank_page.png"]:
    print(f"{f:22s}", image_usable(f))

assert image_usable("receipt.png")[0] is True
assert image_usable("receipt_too_dark.png")[0] is False
assert image_usable("blank_page.png")[0] is False

receipt.png            (True, 'ok (brightness 240, contrast 56)')
receipt_too_dark.png   (False, 'too dark (brightness 19)')
blank_page.png         (False, 'blank or washed out (contrast 0)')


In [13]:
# Basic array handling a vision pipeline needs: shapes, a crop, and a downscale
img = Image.open(path("receipt.png"))
arr = np.asarray(img)
print("Full image array shape (height, width, channels):", arr.shape)

status_line = img.crop((0, 215, 480, 275))
print("Cropped status-line region:", status_line.size)

def downscale(image, max_side=256):
    scale = max_side / max(image.size)
    return image if scale >= 1 else image.resize((int(image.width * scale), int(image.height * scale)))

small = downscale(img)
print("Downscaled for a model:", small.size)
assert max(small.size) <= 256

Full image array shape (height, width, channels): (300, 480, 3)
Cropped status-line region: (480, 60)
Downscaled for a model: (256, 160)


An OCR or vision-language model would run next, **only on images that pass the gate**. None is installed here, so this notebook stops at the gate rather than pretend to read the receipt.

## 8. The orchestrated pipeline — explicit statuses, no silent defaults

In [14]:
def process_voice_note(filename):
    raw = transcribe(filename)
    ok, why = transcript_usable(raw)
    if not ok:
        return {"file": filename, "status": "rejected", "reason": why}

    fixed, corrections = repair_cities(raw)
    fields = extract_fields(fixed)
    base = {"file": filename, "corrections": corrections, **fields}

    if fields["city"] is None or fields["intent"] == "unknown":
        return {**base, "status": "human_review", "reason": "missing city or unknown intent"}
    if fields["intent"] == "refund":
        return {**base, "status": "needs_approval", "reason": "refund touches money — human approves"}
    return {**base, "status": "processed", "reason": "routine request"}

results = pd.DataFrame([process_voice_note(f) for f in
    ["voice_note_1.wav", "voice_note_2.wav", "voice_note_3.wav", "voice_note_silent.wav"]])
results[["file", "status", "city", "order_id", "intent", "corrections", "reason"]]

,file,status,city,order_id,intent,corrections,reason
0,voice_note_1.wav,needs_approval,gurugram,None,refund,"[(guregram, Gurugram)]",refund touches money — human approves
1,voice_note_2.wav,processed,noida,4572,address_change,[],routine request
2,voice_note_3.wav,processed,chennai,None,replacement,[],routine request
3,voice_note_silent.wav,rejected,NaN,NaN,NaN,NaN,mostly non-letter characters (degenerate output)


In [15]:
status = dict(zip(results["file"], results["status"]))
assert status["voice_note_1.wav"] == "needs_approval"     # refund, city repaired to Gurugram
assert status["voice_note_2.wav"] == "processed"          # address change for Noida, order 4572
assert status["voice_note_3.wav"] == "processed"          # replacement (negated refund handled)
assert status["voice_note_silent.wav"] == "rejected"      # degenerate transcript gated
assert results.loc[results.file == "voice_note_2.wav", "order_id"].iloc[0] == "4572"
print("All pipeline routing checks passed.")

All pipeline routing checks passed.


## 9. Optional — a local LLM for extraction, validated like any LLM output

If Ollama is running, ask a local model for the same fields as JSON, then **validate** it exactly as in the Structured Outputs session. If it isn't, this cell explains and skips. Output varies by run.

In [16]:
import json, requests

def ollama_models():
    try:
        return [m['name'] for m in requests.get('http://localhost:11434/api/tags', timeout=3).json()['models']]
    except Exception:
        return []

REQUIRED = {'city': str, 'intent': str}

def valid_extraction(obj):
    return (isinstance(obj, dict) and all(isinstance(obj.get(k), v) for k, v in REQUIRED.items())
            and obj['city'].lower() in CITIES)

assert valid_extraction({'city': 'Noida', 'intent': 'address_change'})
assert not valid_extraction({'city': 'Atlantis', 'intent': 'refund'})
assert not valid_extraction({'intent': 'refund'})

models = ollama_models()
text = repair_cities(transcribe('voice_note_3.wav'))[0]
print('Rule-based extractor:', extract_fields(text))

if not models:
    print('Ollama not running - LLM step skipped; the rule-based extractor above is the fallback.')
else:
    prompt = ('Customer voice note: "' + text + '"\nReturn JSON with keys city (one of Gurugram, Mumbai, '
              'Delhi, Noida, Chennai), order_id (string or null), intent (refund, replacement, address_change).')
    for attempt in range(2):
        r = requests.post('http://localhost:11434/api/generate', timeout=300, json={
            'model': models[0], 'prompt': prompt, 'stream': False, 'format': 'json',
            'options': {'temperature': 0.0, 'num_predict': 120}})
        try:
            obj = json.loads(r.json()['response'])
        except json.JSONDecodeError:
            obj = None
        print(f'LLM attempt {attempt + 1} ({models[0]}):', obj, '-> valid' if valid_extraction(obj) else '-> INVALID')
        if valid_extraction(obj):
            break

Rule-based extractor: {'city': 'chennai', 'order_id': None, 'intent': 'replacement'}


LLM attempt 1 (qwen2.5-coder:7b): {'city': 'Chennai', 'order_id': None, 'intent': 'replacement'} -> valid


## 10. Try it yourself

1. Record or synthesise a fourth voice note mentioning "Mumbai" and an order number. Run it through `process_voice_note`.
2. Make a sentence containing the ordinary word "deli" or "mumbling" and test whether `repair_cities` wrongly changes it at `cutoff=0.75`. What cutoff would you choose, and why?
3. Add a new image gate rule (for example: reject images smaller than 100 px wide) and test it.